<a href="https://colab.research.google.com/github/JorgeMiceli1967/IA-SCRAPPING/blob/main/IA_SCRAPPER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sección nueva

In [ ]:
# ============================================================
# RELEVAMIENTO WEB TEMÁTICO PARA ESTADO DEL ARTE
# Compatible con Google Colab y Python local
# ============================================================
#
# Requisitos:
#   pip install requests pandas openpyxl beautifulsoup4 lxml
#
# Si usás Google Colab:
#   descomentá la línea de instalación de paquetes.
#
# Qué hace:
# - busca resultados web por campo temático usando SerpAPI
# - visita cada URL y extrae texto básico
# - calcula puntaje de relevancia
# - clasifica tipo de actor (gobierno, academia, ONG, etc.)
# - exporta resultados a CSV y Excel
# - genera tablas resumen
#
# IMPORTANTE:
# - Necesitás una API key de SerpAPI: https://serpapi.com/
# - No inventa métricas ni clasificaciones “inteligentes”: usa reglas explícitas
# ============================================================

# !pip install -q requests pandas openpyxl beautifulsoup4 lxml

import re
import sys
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from collections import Counter

# Si corrés en Colab, esto permite descarga automática y acceso a secrets
try:
    from google.colab import files, userdata
    IN_COLAB = True
    API_KEY = userdata.get("SERPAPI_KEY")
except Exception:
    IN_COLAB = False
    API_KEY = None

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

if not API_KEY:
    raise ValueError("No se encontró la clave SERPAPI_KEY en los Secrets de Colab.")

print("Clave cargada correctamente.")


LANGUAGE = "es"
COUNTRY = "es"   # España como foco para algunas búsquedas; podés cambiar a "ar", "us", etc.
RESULTS_PER_QUERY = 10
SLEEP_BETWEEN_SEARCHES = 1.2
SLEEP_BETWEEN_PAGES = 0.8
REQUEST_TIMEOUT = 25
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/122.0 Safari/537.36"
)

OUTPUT_CSV = "relevamiento_estado_del_arte.csv"
OUTPUT_XLSX = "relevamiento_estado_del_arte.xlsx"
OUTPUT_SUMMARY_XLSX = "relevamiento_resumen.xlsx"

VISIT_PAGES = True          # si False, solo usa título/snippet del buscador
MAX_TEXT_LENGTH = 12000     # recorta texto de página para no exagerar
MIN_RELEVANCE_SCORE = 2     # umbral mínimo para conservar resultados
DROP_DUPLICATES_BY_URL = True


# ============================================================
# CAMPOS TEMÁTICOS
# ============================================================

FIELDS = {
    "IA y datos en M.F. (Europa y América)": {
        "keywords": [
            "inteligencia artificial ministerio fiscal",
            "AI prosecutor office digital evidence",
            "inteligencia artificial fiscalía prueba digital",
            "AI criminal justice prosecution service",
            "automatización procesos ministerio público",
            "analítica de datos justicia penal",
            "AI prosecution digital transformation",
            "procesamiento de prueba digital fiscalía",
        ],
        "relevance_terms": [
            "inteligencia artificial", "ai", "machine learning", "algoritmo",
            "analítica de datos", "data analytics", "prueba digital",
            "digital evidence", "fiscalía", "ministerio fiscal",
            "ministerio público", "prosecution", "criminal justice",
            "judicial analytics", "automatización", "proceso penal"
        ],
        "institution_terms": [
            "ministerio fiscal", "ministerio público", "fiscalía",
            "prosecutor", "prosecution service", "attorney general",
            "ministerio de justicia", "court", "tribunal"
        ]
    },

    "Servicios jurídicos sector privado y oferta académica (España)": {
        "keywords": [
            "legal tech España",
            "legal technology Spain AI law",
            "software jurídico inteligencia artificial España",
            "compliance AI software Spain",
            "document review legal AI Spain",
            "máster inteligencia artificial derecho España",
            "posgrado derecho tecnología España",
            "grupo investigación derecho inteligencia artificial España",
            "revista inteligencia artificial derecho España",
        ],
        "relevance_terms": [
            "legal tech", "legal ai", "legal analytics", "compliance",
            "document review", "contract analysis", "litigation analytics",
            "máster", "master", "posgrado", "postgraduate",
            "grupo de investigación", "research group", "law school",
            "facultad de derecho", "revista", "journal", "derecho digital"
        ],
        "institution_terms": [
            "universidad", "university", "facultad de derecho", "law school",
            "research center", "grupo de investigación", "despacho", "firma",
            "legal tech", "empresa", "consultora"
        ]
    },

    "Herramientas de IA": {
        "keywords": [
            "large language models legal applications",
            "herramientas de IA para derecho",
            "AI legal assistant tools",
            "legal AI software",
            "LLM legal research",
            "sistemas expertos jurídicos",
            "AI agent legal workflow",
            "self-hosted legal AI",
            "open source LLM legal",
        ],
        "relevance_terms": [
            "llm", "large language model", "generative ai", "ia generativa",
            "ai agent", "agente de ia", "sistema experto", "expert system",
            "nlp", "natural language processing", "legal ai", "chatbot",
            "open source", "self-hosted", "on premise", "local deployment"
        ],
        "institution_terms": [
            "platform", "software", "tool", "herramienta", "modelo",
            "modelo fundacional", "startup", "empresa", "provider"
        ]
    },

    "Bibliografía y jurisprudencia": {
        "keywords": [
            "artificial intelligence criminal justice research",
            "inteligencia artificial proceso penal artículo",
            "AI judicial decision making paper",
            "machine learning criminal justice journal",
            "jurisprudencia inteligencia artificial proceso penal",
            "AI criminal law review article",
            "AI sentencing research",
            "predictive policing legal scholarship",
        ],
        "relevance_terms": [
            "paper", "research article", "journal", "law review",
            "academic article", "artículo", "revista", "jurisprudencia",
            "case law", "doctrina", "sentencia", "tribunal", "court",
            "legal scholarship", "proceso penal", "criminal law"
        ],
        "institution_terms": [
            "journal", "law review", "tribunal", "court", "revista",
            "repositorio", "repository", "ssrn", "heinonline", "scielo"
        ]
    }
}


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def normalize_space(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_domain(url: str) -> str:
    try:
        return urlparse(url).netloc.lower().replace("www.", "")
    except Exception:
        return ""


def classify_actor_type(domain: str, text: str) -> str:
    """
    Clasificación heurística simple y transparente.
    """
    d = (domain or "").lower()
    t = (text or "").lower()

    if any(x in d for x in [".gov", ".gob", "fiscal", "judiciary", "justice", "poderjudicial", "boe.es"]):
        return "gobierno/justicia"
    if any(x in d for x in [".edu", ".ac.", "universidad", "university"]) or \
       any(x in t for x in ["universidad", "university", "facultad", "faculty", "research group", "grupo de investigación"]):
        return "academia"
    if any(x in d for x in [".org", ".int"]) or \
       any(x in t for x in ["ngo", "ong", "nonprofit", "asociación", "fundación", "foundation"]):
        return "ong/organización"
    if any(x in t for x in ["software", "platform", "plataforma", "startup", "company", "empresa", "provider"]):
        return "empresa/proveedor"
    return "otro"


def count_term_hits(text: str, terms: list[str]) -> int:
    text_lower = (text or "").lower()
    hits = 0
    for term in terms:
        if term.lower() in text_lower:
            hits += 1
    return hits


def score_result(title: str, snippet: str, page_text: str, relevance_terms: list[str], institution_terms: list[str]) -> dict:
    """
    Puntaje simple y explícito.
    """
    title_hits = count_term_hits(title, relevance_terms)
    snippet_hits = count_term_hits(snippet, relevance_terms)
    page_hits = count_term_hits(page_text, relevance_terms)
    institution_hits = count_term_hits(page_text, institution_terms) + count_term_hits(snippet, institution_terms)

    score = (title_hits * 3) + (snippet_hits * 2) + page_hits + institution_hits

    return {
        "hits_titulo": title_hits,
        "hits_snippet": snippet_hits,
        "hits_texto": page_hits,
        "hits_institucionales": institution_hits,
        "puntaje_relevancia": score
    }


def search_serpapi(query: str, api_key: str, num_results: int = 10) -> list[dict]:
    endpoint = "https://serpapi.com/search.json"
    params = {
        "engine": "google",
        "q": query,
        "api_key": api_key,
        "hl": LANGUAGE,
        "gl": COUNTRY,
        "num": num_results,
    }
    response = requests.get(endpoint, params=params, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()
    data = response.json()
    return data.get("organic_results", [])


def fetch_page_text(url: str) -> dict:
    """
    Extrae título, meta description y texto visible básico de una página.
    Si falla, devuelve vacío.
    """
    headers = {"User-Agent": USER_AGENT}
    try:
        r = requests.get(url, headers=headers, timeout=REQUEST_TIMEOUT)
        r.raise_for_status()
        content_type = r.headers.get("Content-Type", "")
        if "text/html" not in content_type:
            return {"page_title": "", "meta_description": "", "page_text": "", "http_status": r.status_code}

        soup = BeautifulSoup(r.text, "lxml")

        # eliminar tags ruidosos
        for tag in soup(["script", "style", "noscript", "svg", "img", "header", "footer", "nav", "aside"]):
            tag.extract()

        page_title = normalize_space(soup.title.get_text()) if soup.title else ""

        meta_description = ""
        md = soup.find("meta", attrs={"name": "description"})
        if md and md.get("content"):
            meta_description = normalize_space(md["content"])

        text = normalize_space(soup.get_text(" "))
        if len(text) > MAX_TEXT_LENGTH:
            text = text[:MAX_TEXT_LENGTH]

        return {
            "page_title": page_title,
            "meta_description": meta_description,
            "page_text": text,
            "http_status": r.status_code
        }
    except Exception:
        return {"page_title": "", "meta_description": "", "page_text": "", "http_status": ""}


def normalize_search_result(field_name: str, query: str, item: dict) -> dict:
    url = item.get("link", "").strip()
    title = normalize_space(item.get("title", ""))
    snippet = normalize_space(item.get("snippet", ""))
    domain = extract_domain(url)

    return {
        "campo": field_name,
        "query": query,
        "titulo_busqueda": title,
        "snippet_busqueda": snippet,
        "url": url,
        "dominio": domain
    }


# ============================================================
# RECOLECCIÓN
# ============================================================

if API_KEY == "PON_AQUI_TU_API_KEY":
    raise ValueError("Reemplazá API_KEY por tu clave real de SerpAPI.")

rows = []

for field_name, field_conf in FIELDS.items():
    keywords = field_conf["keywords"]
    relevance_terms = field_conf["relevance_terms"]
    institution_terms = field_conf["institution_terms"]

    print(f"\n=== Campo: {field_name} ===")

    for query in keywords:
        print(f"Buscando: {query}")
        try:
            results = search_serpapi(query, API_KEY, RESULTS_PER_QUERY)

            for item in results:
                row = normalize_search_result(field_name, query, item)
                if not row["url"]:
                    continue

                page_data = {"page_title": "", "meta_description": "", "page_text": "", "http_status": ""}
                if VISIT_PAGES:
                    page_data = fetch_page_text(row["url"])
                    time.sleep(SLEEP_BETWEEN_PAGES)

                combined_text = " ".join([
                    row["titulo_busqueda"],
                    row["snippet_busqueda"],
                    page_data["page_title"],
                    page_data["meta_description"],
                    page_data["page_text"]
                ])

                score_data = score_result(
                    title=row["titulo_busqueda"],
                    snippet=row["snippet_busqueda"],
                    page_text=page_data["page_text"],
                    relevance_terms=relevance_terms,
                    institution_terms=institution_terms
                )

                actor_type = classify_actor_type(row["dominio"], combined_text)

                final_row = {
                    **row,
                    **page_data,
                    **score_data,
                    "tipo_actor": actor_type
                }
                rows.append(final_row)

        except requests.HTTPError as e:
            print(f"Error HTTP con '{query}': {e}")
        except requests.RequestException as e:
            print(f"Error de conexión con '{query}': {e}")
        except Exception as e:
            print(f"Error inesperado con '{query}': {e}")

        time.sleep(SLEEP_BETWEEN_SEARCHES)


# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(rows)

if df.empty:
    print("No se recuperaron resultados.")
else:
    # deduplicación
    if DROP_DUPLICATES_BY_URL:
        df = df.sort_values(by="puntaje_relevancia", ascending=False)
        df = df.drop_duplicates(subset=["url"]).reset_index(drop=True)

    # filtro por relevancia
    df = df[df["puntaje_relevancia"] >= MIN_RELEVANCE_SCORE].reset_index(drop=True)

    # orden final
    df = df.sort_values(
        by=["campo", "puntaje_relevancia", "tipo_actor"],
        ascending=[True, False, True]
    ).reset_index(drop=True)

    # columnas más útiles primero
    preferred_cols = [
        "campo", "query", "tipo_actor", "puntaje_relevancia",
        "hits_titulo", "hits_snippet", "hits_texto", "hits_institucionales",
        "titulo_busqueda", "page_title", "snippet_busqueda", "meta_description",
        "url", "dominio", "http_status", "page_text"
    ]
    df = df[[c for c in preferred_cols if c in df.columns]]

    print("\nVista previa de resultados:")
    print(df.head(15).to_string(index=False))


    # ========================================================
    # TABLAS RESUMEN
    # ========================================================

    summary_by_field = (
        df.groupby("campo")
          .agg(
              resultados=("url", "count"),
              dominios_unicos=("dominio", "nunique"),
              puntaje_promedio=("puntaje_relevancia", "mean")
          )
          .reset_index()
    )

    summary_by_actor = (
        df.groupby(["campo", "tipo_actor"])
          .size()
          .reset_index(name="cantidad")
          .sort_values(["campo", "cantidad"], ascending=[True, False])
    )

    top_domains = (
        df.groupby(["campo", "dominio"])
          .size()
          .reset_index(name="cantidad")
          .sort_values(["campo", "cantidad"], ascending=[True, False])
    )

    print("\nResumen por campo:")
    print(summary_by_field.to_string(index=False))

    print("\nResumen por campo y tipo de actor:")
    print(summary_by_actor.head(30).to_string(index=False))


    # ========================================================
    # EXPORTACIÓN
    # ========================================================

    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

    with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="resultados", index=False)

    with pd.ExcelWriter(OUTPUT_SUMMARY_XLSX, engine="openpyxl") as writer:
        summary_by_field.to_excel(writer, sheet_name="resumen_campo", index=False)
        summary_by_actor.to_excel(writer, sheet_name="resumen_actor", index=False)
        top_domains.to_excel(writer, sheet_name="top_dominios", index=False)

    print(f"\nArchivos guardados:")
    print(f"- {OUTPUT_CSV}")
    print(f"- {OUTPUT_XLSX}")
    print(f"- {OUTPUT_SUMMARY_XLSX}")

    # Descarga en Colab
    if IN_COLAB:
        files.download(OUTPUT_CSV)
        files.download(OUTPUT_XLSX)
        files.download(OUTPUT_SUMMARY_XLSX)


Clave cargada correctamente.


SystemExit: Prueba de acceso a la clave completada

/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
